# 让系统选择资料来源

资料分散在多个库、数据库或外部工具中时，可以先决定要查哪一类资料。本页验证一个简单做法：问题明确提到强化学习时，只在对应章节中检索，再与整本书检索比较。

如果希望由模型在多种检索方式之间选择，可以继续看[让系统决定怎样检索](让系统决定怎样检索.ipynb)。本页只比较章节范围的选择；多资料来源实验仍缺少第二份真实资料。

In [1]:
import sys
from pathlib import Path

def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / 'data' / 'dataset/manifest.json').is_file():
            return folder
    raise FileNotFoundError('没有找到教程数据目录，请从本节所在目录运行。')

course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import build_bm25_search, load_query_catalog, load_pdf_pages
from common.nontraining_utils import load_annotation

data = load_query_catalog()
cases = {item['id']: item for item in data}
pages = load_pdf_pages()
full_search = build_bm25_search(pages)

# 资料范围是在整理文档时登记的章节，不从问题集的 expected_pages 反推。
source_catalog = {
    'lda_derivation': {
        'terms': ('LDA', '线性判别分析', '广义特征值'),
        'pages': range(41, 45),
        'label': '线性判别分析推导（第 41～44 页）',
    },
    'rule_learning': {
        'terms': ('规则学习', '可解释性'),
        'pages': range(191, 193),
        'label': '规则学习（第 191～192 页）',
    },
    'reinforcement_learning': {
        'terms': ('强化学习', 'Bellman'),
        'pages': range(193, 196),
        'label': '强化学习（第 193～195 页）',
    },
}

def choose_source(question):
    scored = {
        name: sum(term in question for term in info['terms'])
        for name, info in source_catalog.items()
    }
    best = max(scored, key=scored.get)
    return best if scored[best] else None

def rank_and_coverage(results, expected_pages):
    expected = set(expected_pages)
    found = sorted(expected.intersection(item.page for item in results))
    rank = next((index for index, item in enumerate(results, 1) if item.page in expected), None)
    return rank, found

for label, case_id in (
    ('主要问题：先选强化学习资料范围', 'bellman_with_reinforcement_scope'),
    ('复查：先选规则学习资料范围', 'rule_learning_interpretability'),
):
    case = cases[case_id]
    route = choose_source(case['query'])
    source = source_catalog[route]
    routed_pages = [page for page in pages if page['page'] in source['pages']]
    routed_search = build_bm25_search(routed_pages)
    baseline = full_search(case['query'], top_k=3)
    routed = routed_search(case['query'], top_k=3)
    annotation = load_annotation(case['id'])
    baseline_rank, baseline_found = rank_and_coverage(baseline, annotation['expected_pages'])
    routed_rank, routed_found = rank_and_coverage(routed, annotation['expected_pages'])
    print('\n' + label)
    print('问题：', case['query'])
    print('选择来源：', source['label'])
    print('整本书前 3 页：', [item.page for item in baseline], '；必要页：', baseline_found, '；首个必要页排名：', baseline_rank or '未出现')
    print('路由后前 3 页：', [item.page for item in routed], '；必要页：', routed_found, '；首个必要页排名：', routed_rank or '未出现')
    print('路由后的第一条原文摘要：', routed[0].text[:150], '...')
    if case_id == 'bellman_with_reinforcement_scope':
        assert not baseline_found and routed_found == [194]
    else:
        assert baseline_rank == 1 and routed_rank == 1


主要问题：先选强化学习资料范围
问题： 强化学习里当前和未来怎么联系？
选择来源： 强化学习（第 193～195 页）
整本书前 3 页： [2, 104, 59] ；必要页： [] ；首个必要页排名： 未出现
路由后前 3 页： [193, 194, 195] ；必要页： [194] ；首个必要页排名： 2
路由后的第一条原文摘要： 第16 章 强化学习 强化学习作为机器学习的子领域，其本身拥有一套完整的理论体系，以及诸多经典和最新前沿算 法，“西瓜书”该章内容仅可作为综述查阅，若想深究建议查阅其他相关书籍（例如《Easy RL：强化 学习教程》 [1]）进行系统性学习。 16.1 任务与奖赏 本节理解强化学习的定义和相关术语的 ...

复查：先选规则学习资料范围
问题： 规则学习为什么具有良好的可解释性？
选择来源： 规则学习（第 191～192 页）
整本书前 3 页： [191, 151, 14] ；必要页： [191] ；首个必要页排名： 1
路由后前 3 页： [191, 192] ；必要页： [191] ；首个必要页排名： 1
路由后的第一条原文摘要： 第15 章 规则学习 规则学习是“符号主义学习”的代表性方法，用来从训练数据中学到一组能对未见示例进行判别的规 则，形如“如果A 或B，并且C 的条件下，D 满足”这样的形式。因为这种学习方法更加贴合人类从数 据中学到经验的描述，具有非常良好的可解释性，是最早开始研究机器学习的技术之一。 15.1  ...


主要问题直接检索整本书时，前 3 页没有第 194 页；关键词路由选择强化学习章节后，第 194 页进入前 3 条。复查问题整本书和路由后都把第 191 页排在第 1 条，说明路由没有改变原本已经命中的结果。

这说明规则清楚时，按关键词选择章节可以缩小检索范围。范围选错时，后面的检索根本看不到正确资料；来源不确定时应扩大范围或请用户补充，而不是强行选择。由模型选择检索方式的单库实验见[让系统决定怎样检索](让系统决定怎样检索.ipynb)；真正的多资料来源实验仍需要第二份真实资料。



## Multi-Document Agent 的结构和边界

Multi-Document Agent（多资料来源助手）为每个资料源建立独立检索器，再由路由器根据问题和资料源说明选择一个或多个来源。选错来源时，后面的检索看不到正确资料；来源不确定时应扩大范围或返回“无法判断”，不能强行路由。

本页先看“强化学习里当前和未来怎么联系？”，再看“规则学习为什么具有良好的可解释性？”作对照。由于当前只有一份 PDF，本页用章节范围模拟资料来源，证明的是“先缩小资料范围”这一机制；它不是跨文件 Agent，也没有声称有第二个真实知识库。


## 从章节范围选择扩展到多资料来源

本页已运行的 `source_catalog` 把同一本书按章节范围分成多个可选来源，并记录路由、检索页和最终回答的边界。扩展到多文件时，应为每个来源保留独立检索器，记录每个来源的命中率和权限过滤，再合并证据；来源无法判断时宁可扩大范围或请用户补充，也不要强行选择。下面的实验结论只覆盖当前 PDF 的章节范围模拟。


## 查询路由与多资料来源选择

查询路由在一次请求开始时根据问题和资料源说明选择检索范围。典型流程是“问题 → 路由器 → 一个或多个资料来源的独立检索器 → 合并证据 → 回答”；路由可以是规则关键词，也可以是模型判断。路由的价值不在多走一步，而在把无关资料排除在后续检索之外。

本页的实验把同一本 `pumpkin_book.pdf` 按章节范围模拟多个资料来源，因此只能证明“先缩小资料范围”这一机制，不是跨文件、跨权限的完整 Multi-Document Agent。在强化学习问题上，整本书前 3 页没有第 194 页，按关键词选择强化学习范围后命中；在规则学习问题上，整库和路由后都把第 191 页排在第 1，说明路由没有改坏原本已命中的结果。

规则路由适合主题词和范围边界稳定的资料；LLM 路由更灵活，但增加调用和误路由风险。选错资料来源后面的检索看不到正确证据，所以来源不确定时应扩大到多个来源、返回无法判断，或请用户补充，不能强行选择。代码中的 `source_catalog` 保存可复查的主题、页范围和标签，`choose_source` 只按问题词选候选；扩展到多资料时还应记录每个来源的检索和生成过程。

本页只为“按关键词选择资料来源（查询路由）”登记上述两道对照问题；不要把本页单 PDF 的章节范围模拟写成已经完成多真实资料库实验。

In [2]:
from common.eval_utils import emit_tutorial_audit

# 统一保存契约：路由和检索完成后才读取 expected_pages。
import json

def _actual_pages(items):
    pages = []
    for item in items:
        page = int(item.page)
        if page not in pages:
            pages.append(page)
    return pages

def _metrics(items, expected_pages):
    pages = _actual_pages(items)
    expected = {int(page) for page in expected_pages}
    found = set(pages) & expected
    rank = next((index for index, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': rank,
            'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def _route_result(case_id):
    case = cases[case_id]
    source = source_catalog[choose_source(case['query'])]
    routed_search = build_bm25_search([page for page in pages if page['page'] in source['pages']])
    baseline = full_search(case['query'], top_k=3)
    routed = routed_search(case['query'], top_k=3)
    # 两条检索已经完成，下面才读取标注并计算排名/覆盖率。
    annotation = load_annotation(case_id)
    expected = annotation['expected_pages']
    return baseline, routed, expected

def _emit(role, case_id, before_items, after_items, purpose=None):
    expected = _route_result(case_id)[2]
    payload = {'case_id': case_id, 'method': '按关键词选择资料来源（Query Routing）', 'role': role,
              'before': _metrics(before_items, expected),
              'after': _metrics(after_items, expected)}
    if purpose:
        payload['check_purpose'] = purpose
    emit_tutorial_audit(payload)

main_baseline, main_routed, _ = _route_result('bellman_with_reinforcement_scope')
check_baseline, check_routed, _ = _route_result('rule_learning_interpretability')
_emit('main', 'bellman_with_reinforcement_scope', main_baseline, main_routed)
_emit('check', 'rule_learning_interpretability', check_baseline, check_routed, '确认没有改坏')


## 从资料路由继续到 GraphRAG

Query Routing 回答“应该去哪个资料范围查”，GraphRAG 回答“实体之间通过哪些关系相连”。两者都可能缩小检索范围，但索引结构不同：本页前面的路由器仍检索原文片段；GraphRAG 会先从资料中构建带来源的 `(subject, relation, object)` 三元组或社区结构，再按问题中的实体遍历相关边，最后把命中的边映回原文证据。

一个可核查的 GraphRAG 流程至少包含：

1. 从原文抽取实体和关系，并为每条边保存 `evidence_id/source/page/quote`；结构化输出不合规时直接拒绝入图。
2. 对同名实体做消歧和合并，保留别名、版本和来源，不把字符串相同直接当成同一对象。
3. 从 query 提取种子实体，在限制跳数、边类型和访问范围后遍历子图。
4. 将关系路径映回连续原文，回答时引用原始 evidence，而不是把图中边本身当成最终事实。
5. 用多跳覆盖率、路径正确性、证据忠实度、延迟和建图成本评估；必须与同语料的向量或混合检索比较。

```python
# 教学接口：edge 必须绑定已经核验的原文证据。
edge = {
    'subject': '交叉验证法',
    'relation': '属于',
    'object': '模型评估方法',
    'evidence_id': '...',
    'source': 'pumpkin_book.pdf',
    'page': 18,
    'quote': '...',
}
# retrieve_subgraph(seed_entities, allowed_relations, max_hops=2)
# -> paths + bound evidence；随后仍要用原文 quote 组成回答上下文。
```

GraphRAG 更适合跨片段关系、多跳路径和全局结构问题；单页定义、精确术语或普通语义匹配通常不需要先建图。当前南瓜书项目没有 canonical 实体关系标注集，因此本节保留方法、数据契约和最小接口，不把静态手写三元组或无法追溯的模型输出伪装成 GraphRAG 效果实验。